# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Working month:** `2026-03` (mid-panel, avoiding the sealed final month). All queries below run against the real Hugging Face warehouse release, not the small starter CSV.


In [1]:
%pip -q install duckdb huggingface_hub

import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Store it as a Colab Secret named HF_TOKEN (key icon, left panel) so this never prompts.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

MONTH_START = "2026-03-01"
MONTH_END   = "2026-04-01"   # exclusive
MID_MONTH   = "2026-03-16"   # split point: first half = features, second half = label

print("Connected. Working month:", MONTH_START, "to", MONTH_END, "(exclusive)")


Connected. Working month: 2026-03-01 to 2026-04-01 (exclusive)


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Contract answer 1 — what one row means:** the raw table `fact_content_daily_performance` has one row per **(page, client, day)** — a daily performance snapshot for one piece of content. For my lane (Refresh / Content Opportunity Scoring), I roll this up: my working unit is **one row = one page (`content_hash_id`), summarized over one month**, because the review queue I'm building ranks pages, not page-days.

**Contract answer 2 — table(s) I'll use:** `fact_content_daily_performance` (the daily fact table) as the main source, plus `dim_clients` to check each client's tracked history before I trust any date window.

**Contract answer 3 — time window:** the mid-panel month **`2026-03-01` to `2026-03-31`** (inclusive), split at `2026-03-16`: the first half (Mar 1–15) builds my features, the second half (Mar 16–31) builds my label. I'm avoiding the final month (June 2026) entirely — that month is a sealed test window, and using it now to develop label logic would mean peeking at the exact kind of future outcome I'm supposed to predict later.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Contract answer 4 — what I'd predict (target/proxy):** `is_declining` — whether a page's total GSC impressions in the second half of March fell by more than 20% compared to the first half. This is a proxy, not a true "the page's health changed" outcome, but it's honestly built: the label only uses information from *after* the feature window closes.

**Contract answer 5 — one thing I deliberately exclude:** `fact_content_query_90d` (the query-level table). It's genuinely useful for my lane later (query concentration, rare/anonymized share), but including it now would make this first data contract too large to verify carefully. I'm keeping this pass to the daily fact table only, and I'll bring query-level signals in as a second contract once this one is proven honest.

| Bucket | Fields | Why |
|---|---|---|
| **Features** (first half of March only) | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (all aggregated Mar 1–15) | Known at the decision point (mid-month), before the label window opens |
| **Label / proxy** | `is_declining` (built from `gsc_impressions`, Mar 16–31 vs Mar 1–15) | The thing I'm trying to predict — built only from the second-half window |
| **Context** (identifiers, not features) | `client_hash_id`, `content_hash_id`, `report_date` | Used for joining and grouping only — the codes themselves carry no signal |
| **Excluded (this pass)** | `fact_content_query_90d`, `dim_content` metadata fields | Kept out on purpose to keep this first contract small and checkable — see contract answer 5 |


In [2]:
# Ground the buckets above in real column names, not guesses.
peek = con.sql(f"SELECT * FROM {TABLES['fact_daily']} LIMIT 3").df()
print("fact_content_daily_performance columns:")
print(list(peek.columns))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_content_daily_performance columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three queries below, in order: **(1) grain** — is one row really one (page, client, day)? **(2) row count and date span** — how big is my March slice, and does it actually cover the dates I claimed? **(3) availability** — using `IS TRUE`, how many rows survive an honest GA4-availability filter?


In [3]:
# Query 1 — Grain: does (report_date, client_hash_id, content_hash_id) uniquely identify a row?
grain_check = con.sql(f"""
    SELECT
        COUNT(*)                                                  AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_keys
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END}'
""").df()

grain_check["grain_confirmed"] = grain_check["total_rows"] == grain_check["distinct_keys"]
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_keys,grain_confirmed
0,9841378,9841378,True


In [4]:
# Query 2 — Row count and date span for my March slice
slice_stats = con.sql(f"""
    SELECT
        COUNT(*)             AS row_count,
        COUNT(DISTINCT content_hash_id) AS distinct_pages,
        COUNT(DISTINCT client_hash_id)  AS distinct_clients,
        MIN(report_date)     AS min_date,
        MAX(report_date)     AS max_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END}'
""").df()

slice_stats


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,distinct_pages,distinct_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [5]:
# Query 3 — Availability: how many March rows survive an honest GA4-availability filter?
avail_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_march_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END}'
""").df()

avail_check["ga4_available_pct"] = (
    avail_check["ga4_available_rows"] / avail_check["total_march_rows"] * 100
).round(1)
avail_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_march_rows,ga4_available_rows,ga4_available_pct
0,9841378,413966,4.2


### Five features (max), each with an "available when?" line

All five are built **only from the first half of March (Mar 1–15)** — before the label window opens on Mar 16.


In [6]:
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_impressions)                                    AS avg_impressions_h1,
        AVG(gsc_clicks)                                         AS avg_clicks_h1,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)       AS ctr_h1,
        AVG(gsc_avg_position)                                   AS avg_position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions_h1
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MID_MONTH}'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 50
""").df()

print(f"{len(features):,} pages with enough first-half volume")
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

92,548 pages with enough first-half volume


,client_hash_id,content_hash_id,avg_impressions_h1,avg_clicks_h1,ctr_h1,avg_position_h1,days_with_impressions_h1
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,28.600000,0.133333,0.004662,4.247255,15
1,client_73cda7b4e4f265ea,content_905aa32a0230694e,5.933333,0.000000,0.000000,3.010741,15
2,client_73cda7b4e4f265ea,content_05434271b257bb68,41.866667,0.066667,0.001592,5.330069,15
3,client_73cda7b4e4f265ea,content_d056587ff7faca0c,85.333333,0.600000,0.007031,4.468441,15
4,client_73cda7b4e4f265ea,content_2662845f598544ef,6.466667,0.000000,0.000000,8.765983,15


**Feature 1 — `avg_impressions_h1`:** knowable at the decision moment (Mar 16) because it only sums impressions from Mar 1–15, which had already happened.

**Feature 2 — `avg_clicks_h1`:** same reasoning — clicks recorded before the decision point.

**Feature 3 — `ctr_h1`:** a ratio of two first-half quantities, so it carries no information from the second half.

**Feature 4 — `avg_position_h1`:** average search position during the first half only — known before Mar 16.

**Feature 5 — `days_with_impressions_h1`:** how many of the 15 first-half days actually had any visibility at all — a consistency signal, also fully first-half.


## 4. The trap: add ONE label-derived column on purpose

First, build the label from the **second half** of March (Mar 16–31) — the window my features above never touch. Then I'll deliberately smuggle a second-half quantity into the feature set and watch the score become suspiciously good.


In [7]:
labels = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_h2
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MID_MONTH}' AND report_date < DATE '{MONTH_END}'
    GROUP BY 1, 2
""").df()

data = features.merge(labels, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining"] = (data["imp_h2"] < 0.8 * data["avg_impressions_h1"] * 15).astype(int)

print(f"{len(data):,} pages with both halves present")
print(f"declining rate: {data['is_declining'].mean()*100:.1f}%")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

92,548 pages with both halves present
declining rate: 28.7%


In [8]:
# THE LEAK — on purpose: include imp_h2 itself, the exact quantity the label is built from.
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["avg_impressions_h1", "avg_clicks_h1", "ctr_h1", "avg_position_h1", "days_with_impressions_h1"]
leaky_features  = honest_features + ["imp_h2"]   # <-- the leak

model_data = data.dropna(subset=leaky_features + ["is_declining"])
X, y = model_data[leaky_features], model_data["is_declining"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

leaky_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
leaky_auc = roc_auc_score(y_te, leaky_model.predict_proba(X_te)[:, 1])
print(f"LEAKY ROC AUC (with imp_h2 included): {leaky_auc:.3f}  <- suspiciously close to perfect")


LEAKY ROC AUC (with imp_h2 included): 0.999  <- suspiciously close to perfect


In [9]:
# Now delete the leak and keep only the honest, first-half-only features.
model_data_honest = data.dropna(subset=honest_features + ["is_declining"])
X, y = model_data_honest[honest_features], model_data_honest["is_declining"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

honest_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f"HONEST ROC AUC (first-half features only): {honest_auc:.3f}")
print()
print(f"Gap: leaky score was {leaky_auc:.3f}, honest score is {honest_auc:.3f} -- "
      f"that gap is exactly what including imp_h2 bought me for free, and it isn't real skill.")


HONEST ROC AUC (first-half features only): 0.614

Gap: leaky score was 0.999, honest score is 0.614 -- that gap is exactly what including imp_h2 bought me for free, and it isn't real skill.


## Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** the panel is **unbalanced across clients** — `dim_clients.gsc_data_start` differs per client, so a single March slice contains a full history of prior months for long-tenured clients, but only partial or no prior history for clients who joined more recently. Any feature that would compare "this March vs an earlier month" (which I'm not building yet, but will want later) is only valid for clients whose `gsc_data_start` is before the window I'd be comparing against — I have to check that per client before trusting a month-over-month comparison, not assume it holds for everyone in the slice.


In [10]:
# Supporting check for the named limitation: how many clients actually had history before March?
history_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_clients,
        COUNT(*) FILTER (WHERE gsc_data_start < DATE '{MONTH_START}') AS clients_with_prior_history
    FROM {TABLES['dim_clients']}
""").df()

history_check


,total_clients,clients_with_prior_history
0,104,52


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
